# CBRP Methodologies Analysis - R Notebook

Análise estatística (Iman-Davenport + Nemenyi + CD-plot + pairwise wins) dos resultados SCBRP, lendo os CSVs em `csv-scbrp/`.

**Como executar:**

1. Rode a célula da seção **1. Setup** (escolha `ANALYSIS_TYPE` = `LB` ou `UB`).
2. Rode as 3 células da seção **2. Análise estatística (CSV SCBRP)** em ordem.
3. (Opcional) seção **3. Análise via Excel** — depende do `.xlsx` original.


## 1. Setup

Execute apenas esta célula para preparar tudo. Para alternar entre LB e UB, mude `ANALYSIS_TYPE` aqui e rode a célula novamente.

In [1]:
# ============================================================================
# 1) SETUP — execute esta célula primeiro. Faz tudo necessário p/ a análise:
#    - carrega libraries
#    - define ANALYSIS_TYPE (LB ou UB) e parâmetros derivados
#    - carrega o CSV SCBRP correspondente
#    - define funções auxiliares (pairwise_win_count, excel_num)
#    - monta o data.frame CSV_BOUNDS_df pronto p/ os testes
# ============================================================================

# --- Libraries ----------------------------------------------------------------
# install.packages("readxl")
# install.packages("xtable")
# install.packages("devtools"); devtools::install_github("b0rxa/scmamp")
suppressPackageStartupMessages({
  library(scmamp)
  library(readxl)
  library(xtable)
})

# --- Configuração: análise das 4 versões do SA (LB only) ---------------------
# O arquivo sa-top4-lb.csv contém apenas LB (uma coluna por versão do SA),
# então força-se ANALYSIS_TYPE = "LB" aqui.
ANALYSIS_TYPE <- "LB"
DATASET_TAG   <- "sa_top4"   # usado p/ nomear PDFs e .tex de saída

BETTER_IS_HIGHER  <- TRUE
MISSING_VALUE     <- 0
RANK_SIGN         <- -1   # rank(-x): maior valor = rank menor (melhor)
PLOTCD_DECREASING <- TRUE

# --- CSV SA (4 versões: SA-Weak, SA-Moderate, SA-Weak-Prep, SA-Moderate-Prep) -
walk_lb_path <- "/home/carlos/Documentos/cbrp-methodologies/latex-tables-sa/sa-top4-lb.csv"
walk_lb_csv  <- read.csv(walk_lb_path, stringsAsFactors = FALSE, check.names = FALSE)

walk_lb_instances    <- walk_lb_csv[[1]]
walk_lb_methods_orig <- names(walk_lb_csv)[-1]
walk_lb_methods      <- gsub("\\.", "-", walk_lb_methods_orig)
walk_lb_best <- lapply(walk_lb_methods_orig, function(col) {
  vals <- walk_lb_csv[[col]]; names(vals) <- walk_lb_instances; vals
})
names(walk_lb_best) <- walk_lb_methods

# --- Funções auxiliares ------------------------------------------------------
pairwise_win_count <- function(df, win_type = c("highest", "lowest"),
                               approaches_to_compare = NULL) {
  win_type <- match.arg(win_type)
  if (!is.null(approaches_to_compare)) {
    approaches <- intersect(approaches_to_compare, colnames(df))
    sub_df     <- df[, approaches, drop = FALSE]
  } else {
    approaches <- colnames(df); sub_df <- df
  }
  n <- length(approaches)
  win_matrix <- matrix(0, n, n, dimnames = list(approaches, approaches))
  for (i in seq_len(nrow(sub_df))) {
    row_vals <- as.numeric(sub_df[i, ])
    for (a in seq_len(n)) for (b in seq_len(n)) {
      if (a != b && !is.na(row_vals[a]) && !is.na(row_vals[b])) {
        if (win_type == "highest" && row_vals[a] > row_vals[b])
          win_matrix[a, b] <- win_matrix[a, b] + 1
        if (win_type == "lowest"  && row_vals[a] < row_vals[b])
          win_matrix[a, b] <- win_matrix[a, b] + 1
      }
    }
  }
  win_matrix
}

excel_num <- function(x, use_missing_value = TRUE) {
  if (is.numeric(x)) return(x)
  x <- trimws(as.character(x))
  if (use_missing_value && exists("MISSING_VALUE")) {
    x[x == "-"] <- as.character(MISSING_VALUE)
  }
  x[x == ""] <- NA
  has_dot <- grepl("\\.", x); has_comma <- grepl(",", x)
  y <- x
  y[has_dot & has_comma] <- gsub("\\.", "", y[has_dot & has_comma])
  y[has_comma]           <- gsub(",", ".", y[has_comma])
  suppressWarnings(as.numeric(y))
}

# --- Monta o data.frame de análise (instâncias x métodos) --------------------
CSV_BOUNDS_df <- data.frame(row.names = walk_lb_instances)
for (m in walk_lb_methods) {
  CSV_BOUNDS_df[[m]] <- unname(walk_lb_best[[m]][walk_lb_instances])
}
CSV_BOUNDS_df <- CSV_BOUNDS_df[rowSums(!is.na(CSV_BOUNDS_df)) > 0, , drop = FALSE]
CSV_BOUNDS_df[is.na(CSV_BOUNDS_df)] <- MISSING_VALUE

# --- Resumo ------------------------------------------------------------------
cat("=== SETUP CONCLUÍDO ===\n")
cat("Dataset       :", DATASET_TAG, "\n")
cat("ANALYSIS_TYPE :", ANALYSIS_TYPE, "(", ifelse(BETTER_IS_HIGHER, "maior é melhor", "menor é melhor"), ")\n")
cat("CSV carregado :", walk_lb_path, "\n")
cat("Métodos (k)   :", ncol(CSV_BOUNDS_df), "->", paste(colnames(CSV_BOUNDS_df), collapse = ", "), "\n")
cat("Instâncias (N):", nrow(CSV_BOUNDS_df), "\n\n")
print(head(CSV_BOUNDS_df, 3))


=== SETUP CONCLUÍDO ===
Dataset       : sa_top4 
ANALYSIS_TYPE : LB ( maior é melhor )
CSV carregado : /home/carlos/Documentos/cbrp-methodologies/latex-tables-sa/sa-top4-lb.csv 
Métodos (k)   : 4 -> SA-Weak, SA-Moderate, SA-Weak-Prep, SA-Moderate-Prep 
Instâncias (N): 36 

                 SA-Weak SA-Moderate SA-Weak-Prep SA-Moderate-Prep
alto-santo-500-1  171.92      171.92       171.92           171.92
alto-santo-500-2   65.60       65.60        65.60            65.60
alto-santo-500-3  106.18      106.18       106.18           106.18


## 2. Análise estatística (CSV SCBRP)

Rode as três células abaixo em ordem após o Setup.

### 2.1 Testes estatísticos (Iman-Davenport + Nemenyi + ranks médios)

In [2]:
# Statistical tests for CSV methods
# Iman-Davenport test and Nemenyi post-hoc test

if (ncol(CSV_BOUNDS_df) >= 2) {
  cat("\n=== STATISTICAL TESTS FOR CSV", ANALYSIS_TYPE, "===\n")
  cat("Analysis Type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n")
  
  # Critical value calculation
  k_CSV <- ncol(CSV_BOUNDS_df)
  N_CSV <- nrow(CSV_BOUNDS_df)
  CriticalValue_CSV <- qf(0.95, k_CSV - 1, (k_CSV - 1) * (N_CSV - 1))
  cat("Critical Value (F-distribution, alpha=0.05):", CriticalValue_CSV, "\n\n")
  
  # Iman-Davenport Test
  cat("--- Iman-Davenport Test ---\n")
  res_id_csv_bounds <- imanDavenportTest(CSV_BOUNDS_df)
  print(res_id_csv_bounds[['statistic']])
  cat("\nInterpretation: If statistic >", CriticalValue_CSV, "there are significant differences\n")
  
  # Nemenyi Post-hoc Test
  cat("\n--- Nemenyi Post-hoc Test ---\n")
  res_nemenyi_csv_bounds <- nemenyiTest(CSV_BOUNDS_df)
  print(res_nemenyi_csv_bounds[['statistic']])
  cat("\nCritical Difference:", res_nemenyi_csv_bounds[['statistic']], "\n")
  cat("Methods with rank differences > CD are significantly different\n")
  
  # Compute and display average ranks
  cat("\n--- Average Ranks (lower rank = better, rank 1 is best) ---\n")
  # Use RANK_SIGN to determine ranking direction based on analysis type
  ranks_matrix_csv <- apply(CSV_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_csv)) ranks_matrix_csv <- matrix(ranks_matrix_csv, nrow = k_CSV)
  ranks_matrix_csv <- t(ranks_matrix_csv)
  avg_ranks_csv <- colMeans(ranks_matrix_csv)
  avg_ranks_sorted <- sort(avg_ranks_csv) # show best (lowest rank) first
  print(avg_ranks_sorted)
  
} else {
  cat("\nNeed at least 2 methods for statistical comparison\n")
}


=== STATISTICAL TESTS FOR CSV LB ===
Analysis Type: Higher is better 
Critical Value (F-distribution, alpha=0.05): 2.691133 

--- Iman-Davenport Test ---
Corrected Friedman's chi-squared 
                        10.62921 

Interpretation: If statistic > 2.691133 there are significant differences

--- Nemenyi Post-hoc Test ---
Critical difference 
          0.7912024 

Critical Difference: 0.7912024 
Methods with rank differences > CD are significantly different

--- Average Ranks (lower rank = better, rank 1 is best) ---
    SA-Weak-Prep          SA-Weak      SA-Moderate SA-Moderate-Prep 
        1.597222         2.666667         2.708333         3.027778 


### 2.2 Critical Difference plot (PDF)

In [3]:
# Generate Critical Difference (CD) plot for CSV methods

if (ncol(CSV_BOUNDS_df) >= 2) {
  cat("\n=== GENERATING CD PLOT FOR CSV METHODS ===\n")
  
  N_CSV <- nrow(CSV_BOUNDS_df)
  k_CSV <- ncol(CSV_BOUNDS_df)
  
  # Compute ranks using configured ranking direction
  ranks_matrix_csv <- apply(CSV_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_csv)) ranks_matrix_csv <- matrix(ranks_matrix_csv, nrow = k_CSV)
  ranks_matrix_csv <- t(ranks_matrix_csv) # N x k
  
  R_j_CSV <- colMeans(ranks_matrix_csv)
  
  # Create labels with average ranks
  approach_labels_csv <- colnames(CSV_BOUNDS_df)
  for (j in 1:k_CSV) {
    approach_labels_csv[j] <- sprintf("%s\n(%.2f)", colnames(CSV_BOUNDS_df)[j], R_j_CSV[j])
  }
  
  CSV_BOUNDS_df_plot <- CSV_BOUNDS_df
  colnames(CSV_BOUNDS_df_plot) <- approach_labels_csv
  
  # Generate PDF plot using configured direction
  tag <- if (exists("DATASET_TAG")) DATASET_TAG else "csv_walk"
  pdf_filename <- paste0("cd_", tag, "_", tolower(ANALYSIS_TYPE), ".pdf")
  pdf(pdf_filename, width=12, height=6)
  plotCD(CSV_BOUNDS_df_plot, cex=1, decreasing=PLOTCD_DECREASING)
  dev.off()
  
  cat("CD plot saved to:", pdf_filename, "\n")
} else {
  cat("\nNeed at least 2 methods to generate CD plot\n")
}


=== GENERATING CD PLOT FOR CSV METHODS ===


CD plot saved to: cd_sa_top4_lb.pdf 


### 2.3 Pairwise wins + tabelas LaTeX

In [4]:
# Pairwise win counts for CSV methods
# Analysis focuses on statistically equivalent methods with the best rank

if (ncol(CSV_BOUNDS_df) >= 2 && exists("pairwise_win_count")) {
  cat("\n=== PAIRWISE WIN ANALYSIS FOR CSV METHODS ===\n")
  
  # Get average ranks and critical difference from previous analysis
  if (exists("res_nemenyi_csv_bounds") && exists("avg_ranks_csv")) {
    CD <- res_nemenyi_csv_bounds[['statistic']]
    
    # Find the best (lowest) rank
    best_rank <- min(avg_ranks_csv)
    best_method <- names(which.min(avg_ranks_csv))
    
    cat("\nBest method (lowest rank):", best_method, "with rank", round(best_rank, 2), "\n")
    cat("Critical Difference (CD):", round(CD, 4), "\n")
    
    # Find all methods statistically equivalent to the best
    # Methods are equivalent if |rank_i - rank_best| <= CD
    equivalent_to_best <- names(avg_ranks_csv[abs(avg_ranks_csv - best_rank) <= CD])
    
    cat("\nMethods statistically equivalent to the best (within CD):\n")
    for (method in equivalent_to_best) {
      cat(sprintf("  - %s (rank: %.2f, diff: %.2f)\n", 
                  method, avg_ranks_csv[method], abs(avg_ranks_csv[method] - best_rank)))
    }
    
    # --- LaTeX table with average ranks (sorted ascending = best first) ---
    library(xtable)
    ranks_tab <- data.frame(
      Rank        = seq_along(avg_ranks_csv),
      Method      = names(sort(avg_ranks_csv)),
      `Avg. rank` = sprintf("%.2f", sort(avg_ranks_csv)),
      Equivalent  = ifelse(names(sort(avg_ranks_csv)) %in% equivalent_to_best, "yes", "no"),
      check.names = FALSE,
      stringsAsFactors = FALSE
    )
    ranks_caption <- sprintf(
      paste0("Average ranks for %s (lower is better). ",
             "Best rank: %.2f (%s). Critical difference (CD) = %.4f; ",
             "entries marked \\texttt{yes} are statistically equivalent to the best."),
      ANALYSIS_TYPE, best_rank, best_method, CD
    )
    cat("\n--- LaTeX Table (Average ranks) ---\n")
    print(xtable(ranks_tab,
                 caption = ranks_caption,
                 label = paste0("tab:csv_avg_ranks_", tolower(ANALYSIS_TYPE)),
                 align = c("l", "r", "l", "r", "c")),
          include.rownames = FALSE, sanitize.text.function = identity)

    tag <- if (exists("DATASET_TAG")) DATASET_TAG else "csv"
    ranks_tex_path <- paste0(tag, "_avg_ranks_", tolower(ANALYSIS_TYPE), ".tex")
    print(xtable(ranks_tab,
                 caption = ranks_caption,
                 label = paste0("tab:csv_avg_ranks_", tolower(ANALYSIS_TYPE)),
                 align = c("l", "r", "l", "r", "c")),
          include.rownames = FALSE,
          sanitize.text.function = identity,
          file = ranks_tex_path,
          booktabs = TRUE)
    cat("LaTeX ranks table written to:", ranks_tex_path, "\n")
    
    # Perform pairwise comparison only among statistically equivalent methods
    cat("\n--- Pairwise Win Analysis (Equivalent Methods Only) ---\n")
    cat("Comparing only the", length(equivalent_to_best), "statistically equivalent methods\n")
    cat("Win type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n\n")
    
    win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
    CSV_pairwise_win_matrix <- pairwise_win_count(CSV_BOUNDS_df, 
                                                    win_type = win_type_param,
                                                    approaches_to_compare = equivalent_to_best)
    cat("Pairwise win counts:\n")
    print(CSV_pairwise_win_matrix)
    
    # Replace diagonal with hyphens for display
    CSV_pairwise_win_matrix_disp <- CSV_pairwise_win_matrix
    diag(CSV_pairwise_win_matrix_disp) <- "-"
    
    cat("\nPairwise win counts [with '-' on diagonal]:\n")
    print(CSV_pairwise_win_matrix_disp)
    
    # Generate LaTeX table
    library(xtable)
    CSV_pairwise_win_matrix_disp_df <- as.data.frame.matrix(CSV_pairwise_win_matrix_disp)
    CSV_pairwise_win_matrix_disp_df[] <- lapply(CSV_pairwise_win_matrix_disp_df, as.character)
    
    caption_text <- paste0("Pairwise win counts for statistically equivalent CSV ", ANALYSIS_TYPE, " approaches.")
    cat("\n--- LaTeX Table (Statistically Equivalent Methods) ---\n")
    print(xtable(CSV_pairwise_win_matrix_disp_df, 
                 caption = caption_text, 
                 label = "tab:csv_pairwise_wins_equiv",
                 align = c("l", rep("c", ncol(CSV_pairwise_win_matrix_disp_df)))),
          include.rownames=TRUE, sanitize.text.function=identity)
    
  } else {
    cat("\nPlease run the statistical tests cell first (Cell 13)\n")
  }
  
  # Also show full comparison for reference
  cat("\n\n=== FULL PAIRWISE COMPARISON (ALL CSV METHODS) ===\n")
  win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
  CSV_pairwise_win_matrix_full <- pairwise_win_count(CSV_BOUNDS_df, win_type = win_type_param)
  cat("Pairwise win counts (all methods):\n")
  print(CSV_pairwise_win_matrix_full)
  
} else if (!exists("pairwise_win_count")) {
  cat("\nPlease run the cell that defines pairwise_win_count function first\n")
} else {
  cat("\nNeed at least 2 methods for pairwise comparison\n")
}


=== PAIRWISE WIN ANALYSIS FOR CSV METHODS ===

Best method (lowest rank): SA-Weak-Prep with rank 1.6 
Critical Difference (CD): 0.7912 

Methods statistically equivalent to the best (within CD):
  - SA-Weak-Prep (rank: 1.60, diff: 0.00)

--- LaTeX Table (Average ranks) ---
% latex table generated in R 4.3.3 by xtable 1.8-4 package
% Sat Apr 18 14:40:37 2026
\begin{table}[ht]
\centering
\begin{tabular}{rlrc}
  \hline
Rank & Method & Avg. rank & Equivalent \\ 
  \hline
  1 & SA-Weak-Prep & 1.60 & yes \\ 
    2 & SA-Weak & 2.67 & no \\ 
    3 & SA-Moderate & 2.71 & no \\ 
    4 & SA-Moderate-Prep & 3.03 & no \\ 
   \hline
\end{tabular}
\caption{Average ranks for LB (lower is better). Best rank: 1.60 (SA-Weak-Prep). Critical difference (CD) = 0.7912; entries marked \texttt{yes} are statistically equivalent to the best.} 
\label{tab:csv_avg_ranks_lb}
\end{table}
LaTeX ranks table written to: sa_top4_avg_ranks_lb.tex 

--- Pairwise Win Analysis (Equivalent Methods Only) ---
Comparing only t

## 3. (Opcional) Análise via Excel

Estas células dependem do arquivo `PhD - Deterministic Trail Models Results.xlsx`. Se você não tem o arquivo, ignore a seção inteira.

In [38]:
# Helper: coerce Excel-imported numeric columns that may come as character
# Handles comma decimals ("12,34"), thousands separators ("1.234,56"), and missing values ("-")
excel_num <- function(x, use_missing_value = TRUE) {
  if (is.numeric(x)) return(x)
  x <- trimws(as.character(x))
  
  # Handle missing values represented as "-"
  if (use_missing_value && exists("MISSING_VALUE")) {
    missing_mask <- x == "-"
    x[missing_mask] <- as.character(MISSING_VALUE)
  }
  
  x[x == ""] <- NA

  has_dot <- grepl("\\.", x)
  has_comma <- grepl(",", x)

  y <- x
  both <- has_dot & has_comma
  y[both] <- gsub("\\.", "", y[both])      # remove thousands '.'
  y[has_comma] <- gsub(",", ".", y[has_comma]) # convert decimal ',' -> '.'

  result <- suppressWarnings(as.numeric(y))
  return(result)
}

In [37]:
# Define path to the Excel file
excel_path <- "/home/carlos/Documentos/cbrp-methodologies/PhD - Deterministic Trail Models Results.xlsx"

# Get all sheet names
sheet_names <- excel_sheets(excel_path)

# Remove the "Compare-all" sheet if it exists
sheet_names <- sheet_names[!(sheet_names == "Compare-all" | grepl("Greedy", sheet_names, ignore.case = TRUE))]

# Clean and standardize method names
clean_names <- gsub("Trail ", "", sheet_names)
clean_names <- trimws(clean_names)

# Rename methods to standard naming convention
clean_names <- gsub("^Exp$", "Path-CBRP", clean_names)
clean_names <- gsub("^Exp Prep$", "Path-CBRP-Prep", clean_names)
clean_names <- gsub("^Exp Frac-Cut$", "Path-CBRP-Frac", clean_names)
clean_names <- gsub("^Exp Frac-Cut Prep$", "Path-CBRP-Frac-Prep", clean_names)
clean_names <- gsub("^MTZ$", "Path-CBRP-MTZ", clean_names)
clean_names <- gsub("^MTZ Prep$", "Path-CBRP-MTZ-Prep", clean_names)

# Read each remaining sheet into a list of dataframes
data_list <- lapply(sheet_names, function(sheet) {
  read_excel(excel_path, sheet = sheet)
})
names(data_list) <- clean_names

cat("\n=== EXCEL DATA LOADED ===\n")
cat("Methods found:", length(clean_names), "\n")
print(clean_names)

# ============================================================================
# BUILD LB AND UB DATAFRAMES FROM EXCEL
# ============================================================================

# Get all unique instances across all sheets
excel_instances <- sort(unique(unlist(lapply(data_list, function(df) df$Instance))))

# Initialize LB and UB dataframes
EXCEL_LB_df <- data.frame(row.names = excel_instances)
EXCEL_UB_df <- data.frame(row.names = excel_instances)

# Extract LB and UB from each method
for (method in names(data_list)) {
  df <- data_list[[method]]
  
  # Create named vectors for LB and UB
  lb_map <- setNames(excel_num(df$LB), df$Instance)
  ub_map <- setNames(excel_num(df$UB), df$Instance)
  
  # Add to dataframes
  EXCEL_LB_df[[method]] <- unname(lb_map[excel_instances])
  EXCEL_UB_df[[method]] <- unname(ub_map[excel_instances])
}

# Fill missing values
# For LB: missing means we don't have a lower bound, use 0
# For UB: missing means we don't have an upper bound, use Inf
EXCEL_LB_df[is.na(EXCEL_LB_df)] <- 0
EXCEL_UB_df[is.na(EXCEL_UB_df)] <- Inf

cat("\n=== EXCEL DATAFRAMES CREATED ===\n")
cat("EXCEL_LB_df dimensions:", nrow(EXCEL_LB_df), "instances x", ncol(EXCEL_LB_df), "methods\n")
cat("EXCEL_UB_df dimensions:", nrow(EXCEL_UB_df), "instances x", ncol(EXCEL_UB_df), "methods\n")
cat("\nMethods in dataframes:\n")
print(colnames(EXCEL_LB_df))
cat("\nFirst few rows of LB:\n")
print(head(EXCEL_LB_df, 3))
cat("\nFirst few rows of UB:\n")
print(head(EXCEL_UB_df, 3))


ERROR: Error: `path` does not exist: ‘/home/carlos/Documentos/cbrp-methodologies/PhD - Deterministic Trail Models Results.xlsx’


In [40]:
# Select the appropriate dataframe based on ANALYSIS_TYPE flag
if (ANALYSIS_TYPE == "LB") {
  EXCEL_BOUNDS_df <- EXCEL_LB_df
  cat("\n=== SELECTED: EXCEL LOWER BOUNDS ===\n")
} else {
  EXCEL_BOUNDS_df <- EXCEL_UB_df
  cat("\n=== SELECTED: EXCEL UPPER BOUNDS ===\n")
}

cat("Analysis Type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n")
cat("Methods:", ncol(EXCEL_BOUNDS_df), "\n")
cat("Instances:", nrow(EXCEL_BOUNDS_df), "\n")
cat("\nMethods available:\n")
print(colnames(EXCEL_BOUNDS_df))
cat("\nFirst few rows:\n")
print(head(EXCEL_BOUNDS_df, 3))

ERROR: Error: objeto 'EXCEL_LB_df' não encontrado


In [23]:
# Statistical tests for EXCEL methods
# Iman-Davenport test and Nemenyi post-hoc test

if (ncol(EXCEL_BOUNDS_df) >= 2) {
  cat("\n=== STATISTICAL TESTS FOR EXCEL", ANALYSIS_TYPE, "===\n")
  cat("Analysis Type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n")
  
  # Critical value calculation
  k_EXCEL <- ncol(EXCEL_BOUNDS_df)
  N_EXCEL <- nrow(EXCEL_BOUNDS_df)
  CriticalValue_EXCEL <- qf(0.95, k_EXCEL - 1, (k_EXCEL - 1) * (N_EXCEL - 1))
  cat("Critical Value (F-distribution, alpha=0.05):", CriticalValue_EXCEL, "\n\n")
  
  # Iman-Davenport Test
  cat("--- Iman-Davenport Test ---\n")
  res_id_excel_bounds <- imanDavenportTest(EXCEL_BOUNDS_df)
  print(res_id_excel_bounds[['statistic']])
  cat("\nInterpretation: If statistic >", CriticalValue_EXCEL, "there are significant differences\n")
  
  # Nemenyi Post-hoc Test
  cat("\n--- Nemenyi Post-hoc Test ---\n")
  res_nemenyi_excel_bounds <- nemenyiTest(EXCEL_BOUNDS_df)
  print(res_nemenyi_excel_bounds[['statistic']])
  cat("\nCritical Difference:", res_nemenyi_excel_bounds[['statistic']], "\n")
  cat("Methods with rank differences > CD are significantly different\n")
  
  # Compute and display average ranks
  cat("\n--- Average Ranks (lower rank = better, rank 1 is best) ---\n")
  # Use RANK_SIGN to determine ranking direction based on analysis type
  ranks_matrix_excel <- apply(EXCEL_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_excel)) ranks_matrix_excel <- matrix(ranks_matrix_excel, nrow = k_EXCEL)
  ranks_matrix_excel <- t(ranks_matrix_excel)
  avg_ranks_excel <- colMeans(ranks_matrix_excel)
  avg_ranks_sorted_excel <- sort(avg_ranks_excel) # show best (lowest rank) first
  print(avg_ranks_sorted_excel)
  
} else {
  cat("\nNeed at least 2 methods for statistical comparison\n")
}

ERROR: Error: objeto 'EXCEL_BOUNDS_df' não encontrado


In [21]:
# Generate Critical Difference (CD) plot for EXCEL methods

if (ncol(EXCEL_BOUNDS_df) >= 2) {
  cat("\n=== GENERATING CD PLOT FOR EXCEL METHODS ===\n")
  
  N_EXCEL <- nrow(EXCEL_BOUNDS_df)
  k_EXCEL <- ncol(EXCEL_BOUNDS_df)
  
  # Compute ranks using configured ranking direction
  ranks_matrix_excel <- apply(EXCEL_BOUNDS_df, 1, function(x) rank(RANK_SIGN * x, ties.method = "average"))
  if (!is.matrix(ranks_matrix_excel)) ranks_matrix_excel <- matrix(ranks_matrix_excel, nrow = k_EXCEL)
  ranks_matrix_excel <- t(ranks_matrix_excel) # N x k
  
  R_j_EXCEL <- colMeans(ranks_matrix_excel)
  
  # Create labels with average ranks
  approach_labels_excel <- colnames(EXCEL_BOUNDS_df)
  for (j in 1:k_EXCEL) {
    approach_labels_excel[j] <- sprintf("%s\n(%.2f)", colnames(EXCEL_BOUNDS_df)[j], R_j_EXCEL[j])
  }
  
  EXCEL_BOUNDS_df_plot <- EXCEL_BOUNDS_df
  colnames(EXCEL_BOUNDS_df_plot) <- approach_labels_excel
  
  # Generate PDF plot using configured direction
  pdf_filename <- paste0("cd_excel_", tolower(ANALYSIS_TYPE), ".pdf")
  pdf(pdf_filename, width=12, height=6)
  plotCD(EXCEL_BOUNDS_df_plot, cex=1, decreasing=PLOTCD_DECREASING)
  dev.off()
  
  cat("CD plot saved to:", pdf_filename, "\n")
} else {
  cat("\nNeed at least 2 methods to generate CD plot\n")
}

ERROR: Error: objeto 'EXCEL_BOUNDS_df' não encontrado


In [24]:
# Pairwise win counts for EXCEL methods
# Analysis focuses on statistically equivalent methods with the best rank

if (ncol(EXCEL_BOUNDS_df) >= 2 && exists("pairwise_win_count")) {
  cat("\n=== PAIRWISE WIN ANALYSIS FOR EXCEL METHODS ===\n")
  
  # Get average ranks and critical difference from previous analysis
  if (exists("res_nemenyi_excel_bounds") && exists("avg_ranks_excel")) {
    CD <- res_nemenyi_excel_bounds[['statistic']]
    
    # Find the best (lowest) rank
    best_rank <- min(avg_ranks_excel)
    best_method <- names(which.min(avg_ranks_excel))
    
    cat("\nBest method (lowest rank):", best_method, "with rank", round(best_rank, 2), "\n")
    cat("Critical Difference (CD):", round(CD, 4), "\n")
    
    # Find all methods statistically equivalent to the best
    # Methods are equivalent if |rank_i - rank_best| <= CD
    equivalent_to_best <- names(avg_ranks_excel[abs(avg_ranks_excel - best_rank) <= CD])
    
    cat("\nMethods statistically equivalent to the best (within CD):\n")
    for (method in equivalent_to_best) {
      cat(sprintf("  - %s (rank: %.2f, diff: %.2f)\n", 
                  method, avg_ranks_excel[method], abs(avg_ranks_excel[method] - best_rank)))
    }
    
    # Perform pairwise comparison only among statistically equivalent methods
    cat("\n--- Pairwise Win Analysis (Equivalent Methods Only) ---\n")
    cat("Comparing only the", length(equivalent_to_best), "statistically equivalent methods\n")
    cat("Win type:", ifelse(BETTER_IS_HIGHER, "Higher is better", "Lower is better"), "\n\n")
    
    win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
    EXCEL_pairwise_win_matrix <- pairwise_win_count(EXCEL_BOUNDS_df, 
                                                    win_type = win_type_param,
                                                    approaches_to_compare = equivalent_to_best)
    cat("Pairwise win counts:\n")
    print(EXCEL_pairwise_win_matrix)
    
    # Replace diagonal with hyphens for display
    EXCEL_pairwise_win_matrix_disp <- EXCEL_pairwise_win_matrix
    diag(EXCEL_pairwise_win_matrix_disp) <- "-"
    
    cat("\nPairwise win counts [with '-' on diagonal]:\n")
    print(EXCEL_pairwise_win_matrix_disp)
    
    # Generate LaTeX table
    library(xtable)
    EXCEL_pairwise_win_matrix_disp_df <- as.data.frame.matrix(EXCEL_pairwise_win_matrix_disp)
    EXCEL_pairwise_win_matrix_disp_df[] <- lapply(EXCEL_pairwise_win_matrix_disp_df, as.character)
    
    caption_text <- paste0("Pairwise win counts for statistically equivalent EXCEL ", ANALYSIS_TYPE, " approaches.")
    cat("\n--- LaTeX Table (Statistically Equivalent Methods) ---\n")
    print(xtable(EXCEL_pairwise_win_matrix_disp_df, 
                 caption = caption_text, 
                 label = "tab:excel_pairwise_wins_equiv",
                 align = c("l", rep("c", ncol(EXCEL_pairwise_win_matrix_disp_df)))),
          include.rownames=TRUE, sanitize.text.function=identity)
    
  } else {
    cat("\nPlease run the statistical tests cell first (Cell 12)\n")
  }
  
  # Also show full comparison for reference
  cat("\n\n=== FULL PAIRWISE COMPARISON (ALL EXCEL METHODS) ===\n")
  win_type_param <- ifelse(BETTER_IS_HIGHER, "highest", "lowest")
  EXCEL_pairwise_win_matrix_full <- pairwise_win_count(EXCEL_BOUNDS_df, win_type = win_type_param)
  cat("Pairwise win counts (all methods):\n")
  print(EXCEL_pairwise_win_matrix_full)
  
} else if (!exists("pairwise_win_count")) {
  cat("\nPlease run the cell that defines pairwise_win_count function first\n")
} else {
  cat("\nNeed at least 2 methods for pairwise comparison\n")
}

ERROR: Error: objeto 'EXCEL_BOUNDS_df' não encontrado


## Exploratory Data Analysis

View and summarize the loaded data.


In [34]:
# IMPORTANT: sheets may have different row orders/filters.
# Build comparison data frames by aligning rows using the 'Instance' key.

# Get all instances from Excel sheets
instances_excel <- sort(unique(unlist(lapply(data_list, function(df) df$Instance))))

# Get instances from CSV
instances_csv <- walk_lb_instances

# Combine all instances (union of both sources)
instances_all <- sort(unique(c(instances_excel, instances_csv)))

cat("\n=== Data Integration Summary ===\n")
cat("Instances from Excel:", length(instances_excel), "\n")
cat("Instances from CSV:", length(instances_csv), "\n")
cat("Total unique instances:", length(instances_all), "\n\n")

# Initialize dataframes
LB_df <- data.frame(row.names = instances_all)
UB_df <- data.frame(row.names = instances_all)

# Add data from Excel sheets (existing code)
for (method in names(data_list)) {
  df <- data_list[[method]]
  # named vectors keyed by Instance
  lb_map <- setNames(excel_num(df$LB), df$Instance)
  ub_map <- setNames(excel_num(df$UB), df$Instance)

  LB_df[[method]] <- unname(lb_map[instances_all])
  UB_df[[method]] <- unname(ub_map[instances_all])
}

# Add data from CSV (lower bounds only)
for (method in walk_lb_methods) {
  lb_vector <- walk_lb_best[[method]]
  # Map CSV values to all instances
  LB_df[[method]] <- unname(lb_vector[instances_all])
}

# Fill missing instances (when a sheet doesn't contain some Instance rows)
LB_df[is.na(LB_df)] <- 0
UB_df[is.na(UB_df)] <- Inf

cat("LB_df dimensions:", nrow(LB_df), "instances x", ncol(LB_df), "methods\n")
cat("UB_df dimensions:", nrow(UB_df), "instances x", ncol(UB_df), "methods\n")
cat("\nMethods in LB_df:\n")
print(colnames(LB_df))
cat("\nMethods in UB_df:\n")
print(colnames(UB_df))


ERROR: Error: objeto 'data_list' não encontrado


In [30]:
# Compute the CriticalValue for LB_df and UB_df using the qf function.
# Normally, for the Iman-Davenport test, the critical value is based on the F-distribution.

# For LB_df
k_LB <- ncol(LB_df)
N_LB <- nrow(LB_df)
CriticalValue_LB <- qf(0.95, k_LB - 1, (k_LB - 1) * (N_LB - 1))
print(paste("Critical Value for LB_df:", CriticalValue_LB))

# For UB_df
k_UB <- ncol(UB_df)
N_UB <- nrow(UB_df)
CriticalValue_UB <- qf(0.95, k_UB - 1, (k_UB - 1) * (N_UB - 1))
print(paste("Critical Value for UB_df:", CriticalValue_UB))


ERROR: Error: objeto 'LB_df' não encontrado


In [ ]:
print("Lower Bound")
res_id_lb = imanDavenportTest(LB_df)
print(res_id_lb[['statistic']])

res_nemenyi_lb = nemenyiTest(LB_df)
print(res_nemenyi_lb[['statistic']])

[1] "Lower Bound"
Corrected Friedman's chi-squared 
                        9.527342 
Critical difference 
          0.9861445 


In [ ]:
print("Upper Bound")
res_id_ub = imanDavenportTest(UB_df)
print(res_id_ub[['statistic']])

res_nemenyi_ub = nemenyiTest(UB_df)
print(res_nemenyi_ub[['statistic']])

[1] "Upper Bound"
Corrected Friedman's chi-squared 
                        17.95701 
Critical difference 
          0.9861445 


## Visualization

Create plots and visualizations of the data.


In [ ]:
# ==== Plot for LB_df ====
N_LB <- nrow(LB_df)
k_LB <- ncol(LB_df)

ranks_matrix_LB <- apply(LB_df, 1, rank)
if (!is.matrix(ranks_matrix_LB)) ranks_matrix_LB <- matrix(ranks_matrix_LB, nrow = k_LB)
ranks_matrix_LB <- t(ranks_matrix_LB) # N x k

R_j_LB <- colMeans(ranks_matrix_LB)

approach_labels_LB <- colnames(LB_df)
for (j in 1:k_LB) {
  approach_labels_LB[j] <- sprintf("%s\n(%.2f)", colnames(LB_df)[j], R_j_LB[j])
}
LB_df_plot <- LB_df
colnames(LB_df_plot) <- approach_labels_LB

pdf("cd_walk_models_lb.pdf", width=12, height=6)
plotCD(LB_df_plot, cex=1, decreasing=FALSE)
dev.off()

# ==== Plot for UB_df ====
N_UB <- nrow(UB_df)
k_UB <- ncol(UB_df)

ranks_matrix_UB <- apply(UB_df, 1, rank)
if (!is.matrix(ranks_matrix_UB)) ranks_matrix_UB <- matrix(ranks_matrix_UB, nrow = k_UB)
ranks_matrix_UB <- t(ranks_matrix_UB) # N x k

R_j_UB <- colMeans(ranks_matrix_UB)

approach_labels_UB <- colnames(UB_df)
for (j in 1:k_UB) {
  approach_labels_UB[j] <- sprintf("%s\n(%.2f)", colnames(UB_df)[j], R_j_UB[j])
}
UB_df_plot <- UB_df
colnames(UB_df_plot) <- approach_labels_UB

pdf("cd_walk_models_ub.pdf", width=12, height=6)
plotCD(UB_df_plot, cex=1, decreasing=FALSE)
dev.off()


agg_record_297074224 
                   2

agg_record_297074224 
                   2

In [ ]:
pairwise_win_count <- function(df, win_type = c("highest", "lowest"), approaches_to_compare = NULL) {
  win_type <- match.arg(win_type)
  # Select only the approaches to be compared, or all if approaches_to_compare not provided
  if (!is.null(approaches_to_compare)) {
    approaches <- intersect(approaches_to_compare, colnames(df))
    sub_df <- df[, approaches, drop = FALSE]
  } else {
    approaches <- colnames(df)
    sub_df <- df
  }
  n_approaches <- length(approaches)
  win_matrix <- matrix(0, nrow = n_approaches, ncol = n_approaches,
                       dimnames = list(approaches, approaches))
  
  for (i in 1:nrow(sub_df)) {
    row_vals <- as.numeric(sub_df[i, ])
    for (a in 1:n_approaches) {
      for (b in 1:n_approaches) {
        # Do not compare same approach and only compare non-NA pairs
        if (a != b && !is.na(row_vals[a]) && !is.na(row_vals[b])) {
          if (win_type == "highest" && (row_vals[a] > row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
          if (win_type == "lowest" && (row_vals[a] < row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
        }
      }
    }
  }
  # Return the square matrix directly (not in melted data frame form)
  return(win_matrix)
}


In [33]:
# Pairwise win counts for Lower Bounds (higher is better)
# Use all methods available in LB_df
cat("\n=== PAIRWISE WIN ANALYSIS FOR LOWER BOUNDS ===\n")
cat("Comparing all", ncol(LB_df), "methods in LB_df\n\n")

LB_pairwise_win_matrix <- pairwise_win_count(LB_df, win_type = "highest")
print("Pairwise win counts for LB (Higher is better):")
print(LB_pairwise_win_matrix)

# Replace zeros on the diagonal with hyphen "-"
LB_pairwise_win_matrix_disp <- LB_pairwise_win_matrix
diag(LB_pairwise_win_matrix_disp) <- "-"
print("\nPairwise win counts for LB (Higher is better) [with '-' on diagonal]:")
print(LB_pairwise_win_matrix_disp)

# Generate LaTeX table
library(xtable)
# Convert matrix with hyphens to data.frame for xtable compatibility
LB_pairwise_win_matrix_disp_df <- as.data.frame.matrix(LB_pairwise_win_matrix_disp)
# xtable will attempt to coerce numeric columns; force all to character to preserve "-"
LB_pairwise_win_matrix_disp_df[] <- lapply(LB_pairwise_win_matrix_disp_df, as.character)
cat("\nWin counts Among Model LBs (LaTeX table):\n")
print(xtable(LB_pairwise_win_matrix_disp_df, 
             caption = "Win counts Among Model LBs.", 
             align = c("l", rep("c", ncol(LB_pairwise_win_matrix_disp_df)))),
      include.rownames=TRUE, sanitize.text.function=identity)


=== PAIRWISE WIN ANALYSIS FOR LOWER BOUNDS ===


ERROR: Error: objeto 'LB_df' não encontrado


In [ ]:
# Pairwise win counts for Upper Bounds (lower is better)
# Use all methods available in UB_df
cat("\n=== PAIRWISE WIN ANALYSIS FOR UPPER BOUNDS ===\n")
cat("Comparing all", ncol(UB_df), "methods in UB_df\n\n")

UB_pairwise_win_matrix <- pairwise_win_count(UB_df, win_type = "lowest")
print("Pairwise win counts for UB (Lower is better):")
print(UB_pairwise_win_matrix)

# Replace zeros on the diagonal with hyphen "-"
UB_pairwise_win_matrix_disp <- UB_pairwise_win_matrix
diag(UB_pairwise_win_matrix_disp) <- "-"
print("\nPairwise win counts for UB (Lower is better) [with '-' on diagonal]:")
print(UB_pairwise_win_matrix_disp)

# Generate LaTeX table
library(xtable)
# Convert matrix with hyphens to data.frame for xtable compatibility
UB_pairwise_win_matrix_disp_df <- as.data.frame.matrix(UB_pairwise_win_matrix_disp)
# xtable will attempt to coerce numeric columns; force all to character to preserve "-"
UB_pairwise_win_matrix_disp_df[] <- lapply(UB_pairwise_win_matrix_disp_df, as.character)
cat("\nWin counts Among Model UBs (LaTeX table):\n")
print(xtable(UB_pairwise_win_matrix_disp_df, 
             caption = "Win counts Among Model UBs.", 
             align = c("l", rep("c", ncol(UB_pairwise_win_matrix_disp_df)))),
      include.rownames=TRUE, sanitize.text.function=identity)

[1] "Pairwise win counts for UB (Lower is better):"
                   Path-CBRP-MTZ Path-CBRP-MTZ-Prep Path-CBRP-Prep
Path-CBRP-MTZ                  0                  2             14
Path-CBRP-MTZ-Prep            13                  0             19
Path-CBRP-Prep                 1                  0              0
[1] "Pairwise win counts for LB (Lower is better) [with '-' on diagonal]:"
                   Path-CBRP-MTZ Path-CBRP-MTZ-Prep Path-CBRP-Prep
Path-CBRP-MTZ      "-"           "2"                "14"          
Path-CBRP-MTZ-Prep "13"          "-"                "19"          
Path-CBRP-Prep     "1"           "0"                "-"           
Win counts Among Model UBs
% latex table generated in R 4.5.2 by xtable 1.8-4 package
% Thu Nov  6 21:14:50 2025
\begin{table}[ht]
\centering
\begin{tabular}{lccc}
  \hline
 & Path-CBRP-MTZ & Path-CBRP-MTZ-Prep & Path-CBRP-Prep \\ 
  \hline
Path-CBRP-MTZ & - & 2 & 14 \\ 
  Path-CBRP-MTZ-Prep & 13 & - & 19 \\ 
  Path-CBRP-Prep & 1 & 0 &

In [ ]:
# ============================================================================
# COMPREHENSIVE ANALYSIS FOR COMPUTATIONAL EXPERIMENTS SECTION
# ============================================================================

# Helper function to get runtime (handles different column names)
get_runtime <- function(df) {
  runtime_col <- intersect(c("Runtime (s)", "Time (s)"), colnames(df))
  if (length(runtime_col) > 0) return(df[[runtime_col[1]]])
  return(rep(NA, nrow(df)))
}

# Helper function to get attended blocks
get_attended <- function(df) {
  blocks_col <- intersect(c("Attended Blocks", "Attended"), colnames(df))
  if (length(blocks_col) > 0) return(df[[blocks_col[1]]])
  return(rep(NA, nrow(df)))
}

# ============================================================================
# COMPUTE KEY METRICS FOR EACH METHODOLOGY
# ============================================================================

compute_metrics <- function(df, method_name) {
  metrics <- list(method = method_name)
  metrics$n_instances <- nrow(df)
  
  # Optimal solutions (gap == 0)
  metrics$optimal_count <- sum(df[["gap (%)"]] == 0, na.rm = TRUE)
  metrics$optimal_pct <- round(100 * metrics$optimal_count / metrics$n_instances, 1)
  
  # Gap statistics
  metrics$avg_gap <- round(mean(df[["gap (%)"]], na.rm = TRUE), 4)
  metrics$max_gap <- round(max(df[["gap (%)"]], na.rm = TRUE), 4)
  
  # LB and UB
  metrics$avg_LB <- round(mean(df$LB, na.rm = TRUE), 2)
  metrics$avg_UB <- round(mean(df$UB, na.rm = TRUE), 2)
  
  # Runtime
  runtimes <- get_runtime(df)
  metrics$avg_runtime <- round(mean(runtimes, na.rm = TRUE), 2)
  metrics$max_runtime <- round(max(runtimes, na.rm = TRUE), 2)
  metrics$min_runtime <- round(min(runtimes, na.rm = TRUE), 4)
  
  # Attended blocks
  attended <- get_attended(df)
  metrics$avg_attended <- round(mean(attended, na.rm = TRUE), 1)
  
  # Problem size
  metrics$avg_V <- round(mean(as.numeric(df[["||V||"]]), na.rm = TRUE), 1)
  metrics$avg_A <- round(mean(as.numeric(df[["||A||"]]), na.rm = TRUE), 1)
  metrics$avg_B <- round(mean(as.numeric(df[["||B||"]]), na.rm = TRUE), 1)
  
  return(metrics)
}

# Compute metrics for all methodologies
all_metrics <- lapply(names(data_list), function(name) {
  compute_metrics(data_list[[name]], name)
})
names(all_metrics) <- names(data_list)

# ============================================================================
# DISPLAY SUMMARY TABLE
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("SUMMARY METRICS FOR ALL METHODOLOGIES\n")
cat("===============================================================================\n\n")

# Create summary dataframe
summary_df <- data.frame(
  Method = sapply(all_metrics, function(x) x$method),
  Inst = sapply(all_metrics, function(x) x$n_instances),
  Opt = sapply(all_metrics, function(x) x$optimal_count),
  Opt_Pct = sapply(all_metrics, function(x) paste0(x$optimal_pct, "%")),
  Avg_Gap = sapply(all_metrics, function(x) paste0(x$avg_gap, "%")),
  Avg_Time = sapply(all_metrics, function(x) x$avg_runtime),
  Avg_Attend = sapply(all_metrics, function(x) x$avg_attended),
  Avg_LB = sapply(all_metrics, function(x) x$avg_LB),
  Avg_UB = sapply(all_metrics, function(x) x$avg_UB),
  stringsAsFactors = FALSE
)
rownames(summary_df) <- NULL
print(summary_df)

# ============================================================================
# PREPROCESSING IMPACT ANALYSIS
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("PREPROCESSING IMPACT ANALYSIS\n")
cat("===============================================================================\n")

# Path-CBRP vs Path-CBRP-Prep
cat("\n--- Path-CBRP vs Path-CBRP-Prep (Exponential Formulation) ---\n")
exp <- data_list[["Path-CBRP"]]
exp_prep <- data_list[["Path-CBRP-Prep"]]

exp_V <- mean(as.numeric(exp[["||V||"]]), na.rm = TRUE)
exp_prep_V <- mean(as.numeric(exp_prep[["||V||"]]), na.rm = TRUE)
cat("Problem Size |V|:", round(exp_V, 1), "->", round(exp_prep_V, 1), 
    "(", round(100*(1-exp_prep_V/exp_V), 1), "% reduction)\n")

exp_time <- mean(get_runtime(exp), na.rm = TRUE)
exp_prep_time <- mean(get_runtime(exp_prep), na.rm = TRUE)
cat("Avg Runtime:", round(exp_time, 2), "s ->", round(exp_prep_time, 2), "s",
    "(", round(100*(1-exp_prep_time/exp_time), 1), "% reduction)\n")

exp_gap <- mean(exp[["gap (%)"]], na.rm = TRUE)
exp_prep_gap <- mean(exp_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(exp_gap, 4), "% ->", round(exp_prep_gap, 4), "%\n")

# Path-CBRP-MTZ vs Path-CBRP-MTZ-Prep
cat("\n--- Path-CBRP-MTZ vs Path-CBRP-MTZ-Prep (MTZ Formulation) ---\n")
mtz <- data_list[["Path-CBRP-MTZ"]]
mtz_prep <- data_list[["Path-CBRP-MTZ-Prep"]]

mtz_time <- mean(get_runtime(mtz), na.rm = TRUE)
mtz_prep_time <- mean(get_runtime(mtz_prep), na.rm = TRUE)
cat("Avg Runtime:", round(mtz_time, 2), "s ->", round(mtz_prep_time, 2), "s",
    "(", round(100*(1-mtz_prep_time/mtz_time), 1), "% reduction)\n")

mtz_gap <- mean(mtz[["gap (%)"]], na.rm = TRUE)
mtz_prep_gap <- mean(mtz_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(mtz_gap, 4), "% ->", round(mtz_prep_gap, 4), "%\n")

# Path-CBRP-Frac vs Path-CBRP-Frac-Prep
cat("\n--- Path-CBRP-Frac vs Path-CBRP-Frac-Prep (Fractional Cuts) ---\n")
frac <- data_list[["Path-CBRP-Frac"]]
frac_prep <- data_list[["Path-CBRP-Frac-Prep"]]

frac_gap <- mean(frac[["gap (%)"]], na.rm = TRUE)
frac_prep_gap <- mean(frac_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(frac_gap, 4), "% ->", round(frac_prep_gap, 4), "% (WORSENED)\n")

frac_opt <- sum(frac[["gap (%)"]] == 0, na.rm = TRUE)
frac_prep_opt <- sum(frac_prep[["gap (%)"]] == 0, na.rm = TRUE)
cat("Optimal solutions:", frac_opt, "->", frac_prep_opt, "\n")

# ============================================================================
# GENERATE LATEX TABLE FOR PAPER
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("LATEX TABLE - COMPUTATIONAL RESULTS\n")
cat("===============================================================================\n\n")

library(xtable)
latex_df <- summary_df
colnames(latex_df) <- c("Method", "Inst.", "Opt.", "Opt.(%)", "Avg.Gap(%)", 
                         "Avg.Time(s)", "Avg.Blocks", "Avg.LB", "Avg.UB")
print(xtable(latex_df, 
             caption = "Summary of computational results for all methodologies.", 
             label = "tab:summary_results",
             align = c("l", "l", rep("c", 8))),
      include.rownames = FALSE,
      sanitize.text.function = identity)